In [140]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

CSV_PATH = "cgm_raw_data.csv"  # change if needed

DATE_COL = "day__date"
PERSON_COL = "day__client_id__v000"

HARD_MIN, HARD_MAX = 2.0, 25.0     # remove outside this
SOFT_LOW, SOFT_HIGH = 3.0, 15.0    # flag outside this (but keep)
# ---------------------------------------------------

df = pd.read_csv(CSV_PATH)

# 1) Identify time columns robustly (handles '00:00'...'23:55')
time_cols = [c for c in df.columns if re.fullmatch(r"\d{2}:\d{2}", str(c))]
if not time_cols:
    raise ValueError("No time columns found matching HH:MM (e.g., '00:00').")

# 2) Keep only the necessary id/date columns + time columns
keep_cols = [DATE_COL, PERSON_COL] + time_cols
missing = [c for c in [DATE_COL, PERSON_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

wide = df[keep_cols].copy()

# 3) Parse date
wide[DATE_COL] = pd.to_datetime(wide[DATE_COL], errors="coerce").dt.date
wide = wide.dropna(subset=[DATE_COL, PERSON_COL])

# 4) Melt to long
long = wide.melt(
    id_vars=[DATE_COL, PERSON_COL],
    value_vars=time_cols,
    var_name="time",
    value_name="glucose_mmol_l"
)

# 5) Make datetime (calendar day + time)
# (datetime64[ns] for easy rolling windows later)
long["datetime"] = pd.to_datetime(
    long[DATE_COL].astype(str) + " " + long["time"],
    errors="coerce"
)

# 6) Coerce glucose to numeric
long["glucose_mmol_l"] = pd.to_numeric(long["glucose_mmol_l"], errors="coerce")

# 7) Drop empty reads (NaN glucose) — you said these exist
long = long.dropna(subset=["datetime", "glucose_mmol_l"])

# 8) Apply bounds
long["is_hard_outlier"] = (long["glucose_mmol_l"] < HARD_MIN) | (long["glucose_mmol_l"] > HARD_MAX)
long = long.loc[~long["is_hard_outlier"]].copy()

long["is_soft_extreme"] = (long["glucose_mmol_l"] < SOFT_LOW) | (long["glucose_mmol_l"] > SOFT_HIGH)

# 9) Rename/standardize output columns
long = long.rename(columns={
    PERSON_COL: "person_id",
    DATE_COL: "date"
})[["person_id", "date", "datetime", "glucose_mmol_l", "is_soft_extreme"]]

# 10) Sort (important for gap detection / rolling features)
long = long.sort_values(["person_id", "datetime"]).reset_index(drop=True)

# ---- Quick sanity checks (minimal, but catches most issues) ----
# step size stats
dt = long.groupby("person_id")["datetime"].diff()
median_step_min = dt.dt.total_seconds().div(60).median()
print("Rows (long):", len(long))
print("People:", long["person_id"].nunique())
print("Median step (min):", median_step_min)

# expected max per day per person if complete: 288
per_day_counts = long.groupby(["person_id", "date"]).size()
print("Per-day rows (min/median/max):",
      int(per_day_counts.min()), int(per_day_counts.median()), int(per_day_counts.max()))

Rows (long): 60318
People: 20
Median step (min): 5.0
Per-day rows (min/median/max): 3 288 288


In [122]:
# Save to CSV
long.to_csv('long_format_data.csv', index=False)

# Display the first few rows of the new DataFrame
print("Long format DataFrame preview:")
display(long.head())

# Get info about the new DataFrame
print("\nDataFrame info:")
print(long.info())


Long format DataFrame preview:


,person_id,date,datetime,glucose_mmol_l,is_soft_extreme
0,SPIKES-001,2026-02-24,2026-02-24 12:35:00,4.8,False
1,SPIKES-001,2026-02-24,2026-02-24 12:40:00,5.1,False
2,SPIKES-001,2026-02-24,2026-02-24 12:45:00,5.3,False
3,SPIKES-001,2026-02-24,2026-02-24 12:50:00,5.2,False
4,SPIKES-001,2026-02-24,2026-02-24 12:55:00,5.2,False



DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60318 entries, 0 to 60317
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   person_id        60318 non-null  object        
 1   date             60318 non-null  object        
 2   datetime         60318 non-null  datetime64[ns]
 3   glucose_mmol_l   60318 non-null  float64       
 4   is_soft_extreme  60318 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), object(2)
memory usage: 1.9+ MB
None


In [142]:
### Spike Analysis

Long_CSV_PATH = "long_format_data.csv"
df = pd.read_csv(Long_CSV_PATH)
df.columns = df.columns.str.strip()  # removes hidden spaces in headers


In [144]:
def find_spikes(df: pd.DataFrame, person_id = "", *,
               glucose_col: str = "glucose_mmol_l",
               dt_col: str = "datetime",
               id_col: str = "person_id",
               date_col: str = "date",
               baseline_lookback_min: int = 60,
               baseline_exclude_last_min: int = 10,
               amp_thresh: float = 1.0,
               end_band: float = 0.3,
               rise_window_min: int = 15,
               rise_rate_thresh: float = 0.02,
               min_above_min: int = 15,
               refractory_min: int = 30,
               sample_step_min: int = 5,
               expected_noise_mad_floor: float = 0.05,
               w_amp: float = 0.55,
               w_slope: float = 0.25,
               w_dur: float = 0.10,
               w_return: float = 0.10,
              ):
    """
    Detect clinically interpretable glucose spike *events* for one client.

    Returns:
      events_df: one row per event with columns
        Day, start_time, end_time, peak_time, p_val, range, peak_glucose, baseline_at_peak
      annotated_df: original client trace with added columns
        baseline, slope_15m, residual, event_id, in_spike_event,
        is_event_start, is_event_end, is_event_peak, p_val_event

    Notes:
      - p_val is a confidence score in [0,1] (not a classical statistical p-value).
      - Assumes approximately regular 5-min sampling; for best results pre-impute short gaps.
    """
    
    # --- filter + sort ---
    if id_col not in df.columns:
        raise KeyError(f"'{id_col}' not found. Available columns: {df.columns.tolist()}")
    
    # Filter for specific person
    d = df.loc[df[id_col].astype(str) == str(person_id)].copy()
    
    # If no data for this person, return empty dataframes
    if d.empty:
        empty_events = pd.DataFrame(columns=["Day", "start_time", "end_time", "peak_time", "p_val", "range",
                                             "peak_glucose", "baseline_at_peak"])
        # Create empty annotated dataframe with expected columns
        empty_annotated = pd.DataFrame(columns=[glucose_col, dt_col, id_col, date_col, 
                                                "baseline", "slope_15m", "residual", 
                                                "event_id", "in_spike_event", "is_event_start", 
                                                "is_event_end", "is_event_peak", "p_val_event"])
        return empty_events, empty_annotated

    # Process datetime columns
    d[dt_col] = pd.to_datetime(d[dt_col])
    if date_col in d.columns:
        d[date_col] = pd.to_datetime(d[date_col]).dt.date
    else:
        d[date_col] = d[dt_col].dt.date

    d = d.sort_values(dt_col).reset_index(drop=True)
    d = d.dropna(subset=[glucose_col]).reset_index(drop=True)

    g = d[glucose_col].to_numpy(dtype=float)
    t = d[dt_col].to_numpy()
    day = d[date_col].to_numpy()

    step = sample_step_min
    idx_per_min = 1.0 / step

    # windows in samples
    lookback_n = int(baseline_lookback_min * idx_per_min)      # 60min -> 12
    excl_n = int(baseline_exclude_last_min * idx_per_min)      # 10min -> 2
    rise_n = int(rise_window_min * idx_per_min)                # 15min -> 3
    min_above_n = int(min_above_min * idx_per_min)             # 15min -> 3
    refractory_n = int(refractory_min * idx_per_min)           # 30min -> 6

    n = len(g)
    
    # Check if we have enough data points
    if n < (lookback_n + rise_n + 1):
        empty_events = pd.DataFrame(columns=["Day", "start_time", "end_time", "peak_time", "p_val", "range",
                                             "peak_glucose", "baseline_at_peak"])
        # Create annotated dataframe with NaN values
        annotated = d.copy()
        annotated["baseline"] = np.nan
        annotated["slope_15m"] = np.nan
        annotated["residual"] = np.nan
        annotated["event_id"] = pd.Series([pd.NA] * len(annotated), dtype="Int64")
        annotated["in_spike_event"] = False
        annotated["is_event_start"] = False
        annotated["is_event_end"] = False
        annotated["is_event_peak"] = False
        annotated["p_val_event"] = np.nan
        return empty_events, annotated

    # --- baseline B_t: median of prior [t-60 : t-10] ---
    B = np.full(n, np.nan)
    for i in range(n):
        start = i - lookback_n
        end = i - excl_n
        if start >= 0 and end > start:
            B[i] = np.nanmedian(g[start:end])

    # --- slope over 15 minutes (rise rate) ---
    slope = np.full(n, np.nan)  # mmol/L/min
    for i in range(n):
        j = i - rise_n
        if j >= 0:
            slope[i] = (g[i] - g[j]) / rise_window_min

    # --- robust noise estimate for confidence scoring ---
    resid_short = g - pd.Series(g).rolling(window=7, center=True, min_periods=3).median().to_numpy()
    mad = np.nanmedian(np.abs(resid_short - np.nanmedian(resid_short)))
    noise = max(1.4826 * mad, expected_noise_mad_floor)

    # Prepare annotated df scaffolding
    annotated = d.copy()
    annotated["baseline"] = B
    annotated["slope_15m"] = slope
    annotated["residual"] = annotated[glucose_col].to_numpy(dtype=float) - B

    annotated["event_id"] = pd.Series([pd.NA] * n, dtype="Int64")
    annotated["in_spike_event"] = False
    annotated["is_event_start"] = False
    annotated["is_event_end"] = False
    annotated["is_event_peak"] = False
    annotated["p_val_event"] = np.nan

    events = []
    i = 0
    in_refractory_until = -1
    event_id = 0

    while i < n:
        # enforce refractory unless we're near baseline
        if i < in_refractory_until:
            if not (np.isfinite(B[i]) and g[i] <= B[i] + end_band):
                i += 1
                continue

        if not (np.isfinite(B[i]) and np.isfinite(slope[i])):
            i += 1
            continue

        # start condition
        amp_ok = (g[i] - B[i]) >= amp_thresh
        slope_ok = slope[i] >= rise_rate_thresh
        if not (amp_ok and slope_ok):
            i += 1
            continue

        # candidate start: walk back to first point above threshold to set start index
        start_i = i
        while start_i - 1 >= 0 and np.isfinite(B[start_i - 1]) and (g[start_i - 1] - B[start_i - 1]) >= amp_thresh:
            start_i -= 1

        # must remain above threshold for min duration
        above_count = 0
        k = start_i
        while k < n and np.isfinite(B[k]) and (g[k] - B[k]) >= amp_thresh:
            above_count += 1
            k += 1
            if above_count >= min_above_n:
                break
        if above_count < min_above_n:
            i += 1
            continue

        # find peak until return-to-baseline condition met
        peak_i = i
        max_g = g[i]
        min_g = g[i]
        j = i
        returned = False
        first_return_i = None

        while j < n:
            if g[j] > max_g:
                max_g = g[j]
                peak_i = j
            if g[j] < min_g:
                min_g = g[j]

            if np.isfinite(B[j]) and g[j] <= B[j] + end_band:
                returned = True
                first_return_i = j
                break
            j += 1

        if not returned:
            i += 1
            continue

        end_i = first_return_i
        event_range = float(max_g - min_g)

        # ---- p_val (confidence score) ----
        peak_amp = float(max_g - B[peak_i]) if np.isfinite(B[peak_i]) else float(max_g - np.nanmedian(B))
        amp_excess = max(0.0, peak_amp - amp_thresh)
        amp_score = 1.0 - np.exp(-(amp_excess / (noise + 1e-9)))  # saturating

        slope_window_end = min(end_i, start_i + int(30 * idx_per_min))
        local_max_slope = np.nanmax(slope[start_i:slope_window_end + 1]) if slope_window_end >= start_i else slope[i]
        slope_excess = max(0.0, float(local_max_slope) - rise_rate_thresh)
        slope_score = 1.0 - np.exp(-(slope_excess / (noise / max(rise_window_min, 1) + 1e-9)))

        # duration above threshold
        above = 0
        for q in range(start_i, end_i + 1):
            if np.isfinite(B[q]) and (g[q] - B[q]) >= amp_thresh:
                above += 1
        dur_min = above * sample_step_min
        dur_score = min(1.0, dur_min / 60.0)  # saturate at 60 min above threshold

        # return quality
        end_gap = float(g[end_i] - (B[end_i] if np.isfinite(B[end_i]) else np.nanmedian(B)))
        if np.isfinite(end_gap):
            return_score = float(np.clip(1.0 - (max(0.0, end_gap) / max(end_band, 1e-9)), 0.0, 1.0))
        else:
            return_score = 0.5

        p_val = float(np.clip(
            w_amp * amp_score +
            w_slope * slope_score +
            w_dur * dur_score +
            w_return * return_score,
            0.0, 1.0
        ))

        # store event row
        events.append({
            "Day": day[start_i],
            "start_time": pd.to_datetime(t[start_i]),
            "end_time": pd.to_datetime(t[end_i]),
            "peak_time": pd.to_datetime(t[peak_i]),
            "p_val": p_val,
            "range": event_range,
            "peak_glucose": float(max_g),
            "baseline_at_peak": float(B[peak_i]) if np.isfinite(B[peak_i]) else np.nan,
        })

        # annotate original trace rows
        annotated.loc[start_i:end_i, "event_id"] = event_id
        annotated.loc[start_i:end_i, "in_spike_event"] = True
        annotated.loc[start_i, "is_event_start"] = True
        annotated.loc[end_i, "is_event_end"] = True
        annotated.loc[peak_i, "is_event_peak"] = True
        annotated.loc[start_i:end_i, "p_val_event"] = p_val

        # refractory
        in_refractory_until = end_i + refractory_n
        i = end_i + 1
        event_id += 1

    events_df = pd.DataFrame(events)
    if not events_df.empty:
        events_df = events_df.sort_values(["Day", "start_time"]).reset_index(drop=True)

    return events_df, annotated


In [146]:
# First, let's check what person_ids are available in the data
# Run the spike detection for SPIKES-001
events_df, annotated_df = find_spikes(df, person_id="SPIKES-001")

print("SPIKE DETECTION RESULTS")
print(f"Number of spikes detected: {len(events_df)}")
print("\nEvents DataFrame:")
print(events_df)

if len(events_df) > 0:
    print("\nFirst detected spike details:")
    print(events_df.iloc[0])
else:
    print("\nNo spikes detected. Let's examine the data pattern...")
    
    # Get SPIKES-001 data
    person_data = df[df['person_id'] == "SPIKES-001"].copy()
    person_data = person_data.sort_values('datetime')
    
    print(f"\nData for SPIKES-001:")
    print(f"Total readings: {len(person_data)}")
    print(f"Date range: {person_data['date'].min()} to {person_data['date'].max()}")
    print(f"Glucose range: {person_data['glucose_mmol_l'].min():.2f} - {person_data['glucose_mmol_l'].max():.2f} mmol/L")
    print(f"Mean glucose: {person_data['glucose_mmol_l'].mean():.2f} mmol/L")
    
    # Calculate 15-minute rolling differences to see potential spikes
    person_data['glucose_15min_diff'] = person_data['glucose_mmol_l'].diff(3)  # 3 * 5min = 15min
    
    print(f"\nLargest 15-minute rise: {person_data['glucose_15min_diff'].max():.2f} mmol/L")
    print(f"Largest 15-minute drop: {person_data['glucose_15min_diff'].min():.2f} mmol/L")
    
    # Check if any points meet the basic spike criteria
    # Calculate baseline (simplified for checking)
    person_data['baseline_simple'] = person_data['glucose_mmol_l'].rolling(window=12, min_periods=1).mean()
    person_data['above_baseline'] = person_data['glucose_mmol_l'] - person_data['baseline_simple']
    
    meets_amp = person_data['above_baseline'] >= 1.0  # amp_thresh
    meets_slope = person_data['glucose_15min_diff'] >= 0.3  # 0.02 * 15 = 0.3
    
    print(f"\nPoints meeting amplitude threshold (>=1.0 above baseline): {meets_amp.sum()}")
    print(f"Points meeting slope threshold (>=0.3 rise over 15min): {meets_slope.sum()}")
    print(f"Points meeting both: {(meets_amp & meets_slope).sum()}")
    
    # Show the highest rise event if any
    if person_data['glucose_15min_diff'].max() > 0.3:
        max_rise_idx = person_data['glucose_15min_diff'].idxmax()
        print(f"\nHighest rise event at {person_data.loc[max_rise_idx, 'datetime']}:")
        print(f"  Glucose: {person_data.loc[max_rise_idx, 'glucose_mmol_l']:.2f} mmol/L")
        print(f"  15-min rise: {person_data.loc[max_rise_idx, 'glucose_15min_diff']:.2f} mmol/L")
        print(f"  Previous glucose (15min before): {person_data.loc[max_rise_idx-3, 'glucose_mmol_l'] if max_rise_idx-3 >= 0 else 'N/A'}")

SPIKE DETECTION RESULTS
Number of spikes detected: 33

Events DataFrame:
           Day          start_time            end_time           peak_time  \
0   2026-02-24 2026-02-24 19:30:00 2026-02-24 20:35:00 2026-02-24 20:15:00   
1   2026-02-25 2026-02-25 20:10:00 2026-02-25 21:20:00 2026-02-25 21:05:00   
2   2026-02-25 2026-02-25 22:35:00 2026-02-25 23:10:00 2026-02-25 22:50:00   
3   2026-02-26 2026-02-26 12:25:00 2026-02-26 13:00:00 2026-02-26 12:40:00   
4   2026-02-26 2026-02-26 16:25:00 2026-02-26 17:00:00 2026-02-26 16:40:00   
5   2026-02-27 2026-02-27 12:05:00 2026-02-27 12:45:00 2026-02-27 12:30:00   
6   2026-02-27 2026-02-27 15:05:00 2026-02-27 15:35:00 2026-02-27 15:15:00   
7   2026-02-28 2026-02-28 12:30:00 2026-02-28 13:15:00 2026-02-28 13:00:00   
8   2026-02-28 2026-02-28 14:50:00 2026-02-28 15:15:00 2026-02-28 14:55:00   
9   2026-02-28 2026-02-28 22:40:00 2026-02-28 23:15:00 2026-02-28 22:45:00   
10  2026-03-01 2026-03-01 18:45:00 2026-03-01 19:25:00 2026-03-01 19:

In [148]:
def run_spikes_all(df, *,
                  id_col="person_id",
                  output_events_csv="spike_events_all_clients.csv",
                  output_annotated_csv="cgm_annotated_all_clients.csv",
                  **find_spikes_kwargs):
    """
    Runs find_spikes for every person_id in df.

    Saves:
      - output_events_csv: one row per spike event (all clients)
      - output_annotated_csv: 5-min CGM rows annotated with event labels (all clients)

    Returns:
      events_all, annotated_all
    """
    df = df.copy()
    df.columns = df.columns.str.strip()

    if id_col not in df.columns:
        raise KeyError(f"'{id_col}' not found. Columns: {df.columns.tolist()}")

    person_ids = df[id_col].dropna().astype(str).unique()

    all_events = []
    all_annotated = []

    for pid in person_ids:
        events_df, annotated_df = find_spikes(df, person_id=pid, id_col=id_col, **find_spikes_kwargs)

        # Ensure the summary table has exactly what you asked for
        if not events_df.empty:
            events_summary = events_df[["Day", "start_time", "end_time", "p_val", "range"]].copy()
        else:
            events_summary = pd.DataFrame(columns=["Day", "start_time", "end_time", "p_val", "range"])

        events_summary.insert(0, id_col, pid)
        all_events.append(events_summary)

        annotated_df = annotated_df.copy()
        annotated_df[id_col] = pid  # keep id for later faceting / filtering
        all_annotated.append(annotated_df)

    events_all = pd.concat(all_events, ignore_index=True)
    annotated_all = pd.concat(all_annotated, ignore_index=True)

    events_all.to_csv(output_events_csv, index=False)
    annotated_all.to_csv(output_annotated_csv, index=False)

    return events_all, annotated_all


# ---- RUN ----
events_all, annotated_all = run_spikes_all(
    df,  # <-- or df2, whichever is your loaded long df
    id_col="person_id",
    output_events_csv="spike_events_all_clients.csv",
    output_annotated_csv="cgm_annotated_all_clients.csv",
)

print("Saved:", "spike_events_all_clients.csv", "and", "cgm_annotated_all_clients.csv")
print(events_all.head())

Saved: spike_events_all_clients.csv and cgm_annotated_all_clients.csv
    person_id         Day          start_time            end_time     p_val  \
0  SPIKES-001  2026-02-24 2026-02-24 19:30:00 2026-02-24 20:35:00  0.375000   
1  SPIKES-001  2026-02-25 2026-02-25 20:10:00 2026-02-25 21:20:00  0.966655   
2  SPIKES-001  2026-02-25 2026-02-25 22:35:00 2026-02-25 23:10:00  0.875000   
3  SPIKES-001  2026-02-26 2026-02-26 12:25:00 2026-02-26 13:00:00  0.875000   
4  SPIKES-001  2026-02-26 2026-02-26 16:25:00 2026-02-26 17:00:00  0.916665   

   range  
0    1.0  
1    2.6  
2    0.9  
3    1.4  
4    1.0  


In [150]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_cgm_with_spikes_save_png(annotated_df: pd.DataFrame,
                                  events_df: pd.DataFrame,
                                  person_id: str,
                                  out_dir: str = "spike_plots",
                                  glucose_col: str = "glucose_mmol_l",
                                  dt_col: str = "datetime",
                                  day_col: str = "date",
                                  dpi: int = 180):
    """
    Saves ONE PNG per day for a given person_id.
    - Shades spike intervals (orange)
    - Marks spike peaks (red dots)
    Requires annotated_df to contain:
      - in_spike_event, is_event_peak (from find_spikes)
    """

    os.makedirs(out_dir, exist_ok=True)

    a = annotated_df.copy()
    a[dt_col] = pd.to_datetime(a[dt_col])
    if day_col in a.columns:
        a[day_col] = pd.to_datetime(a[day_col]).dt.date
    else:
        a[day_col] = a[dt_col].dt.date

    # Ensure events time columns are datetime
    ev = events_df.copy()
    if not ev.empty:
        ev["start_time"] = pd.to_datetime(ev["start_time"])
        ev["end_time"] = pd.to_datetime(ev["end_time"])
        if "peak_time" in ev.columns:
            ev["peak_time"] = pd.to_datetime(ev["peak_time"])
        # day column in events is "Day" in our detector
        if "Day" in ev.columns:
            ev["Day"] = pd.to_datetime(ev["Day"]).dt.date

    for day in sorted(a[day_col].dropna().unique()):
        day_mask = (a[day_col] == day)
        day_df = a.loc[day_mask].sort_values(dt_col)

        fig, ax = plt.subplots(figsize=(14, 4))

        # Glucose trace
        ax.plot(day_df[dt_col], day_df[glucose_col], color="black", lw=1.2, label="Glucose")

        # Optional baseline
        if "baseline" in day_df.columns:
            ax.plot(day_df[dt_col], day_df["baseline"], color="dodgerblue", lw=1.0, alpha=0.7, label="Baseline")

        # Shade spike intervals from events_df for this day
        if not ev.empty:
            # prefer matching via events "Day" if present
            if "Day" in ev.columns:
                ev_day = ev.loc[ev["Day"] == day]
            else:
                # fallback: use start_time date
                ev_day = ev.loc[ev["start_time"].dt.date == day]

            for _, e in ev_day.iterrows():
                ax.axvspan(e["start_time"], e["end_time"], color="orange", alpha=0.25)

        # Highlight peaks as red dots (from annotated_df)
        if "is_event_peak" in day_df.columns:
            peaks = day_df.loc[day_df["is_event_peak"] == True]
            if not peaks.empty:
                ax.scatter(peaks[dt_col], peaks[glucose_col],
                           color="red", s=35, zorder=5, label="Spike peak")

        ax.set_title(f"{person_id} — {day} (spikes shaded, peaks red)")
        ax.set_xlabel("Time")
        ax.set_ylabel("Glucose (mmol/L)")
        ax.legend(loc="upper right")
        plt.tight_layout()

        out_path = os.path.join(out_dir, f"{person_id}_{day}_spikes.png")
        fig.savefig(out_path, dpi=dpi)
        plt.close(fig)

    print(f"Saved daily PNGs to: {out_dir}/")


# Example run for one person
pid = "SPIKES-003"
events_df, annotated_df = find_spikes(df, person_id=pid)  # or df2
plot_cgm_with_spikes_save_png(annotated_df, events_df, person_id=pid, out_dir="spike_plots")

Saved daily PNGs to: spike_plots/
